In [2]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("Files/Bronze/HR_Employee_Data.csv")

# Standardize column names
df = df.toDF(*[c.replace(" ", "_").lower() for c in df.columns])

# Convert data types
from pyspark.sql.functions import when, col, to_date

df = spark.read.format("csv") \
    .option("header", "true") \
    .load("Files/Bronze/HR_Employee_Data.csv")

# Standardize column names
df = df.toDF(*[c.replace(" ", "_").lower() for c in df.columns])

# Convert data types
df = df.withColumn("hiredate", to_date(col("hiredate"), "yyyy-MM-dd")) \
       .withColumn("age", col("age").cast("int")) \
       .withColumn("monthlyincome", col("monthlyincome").cast("int")) \
       .withColumn("yearsatcompany", col("yearsatcompany").cast("int")) \
       .withColumn("yearsincurrentrole", col("yearsincurrentrole").cast("int"))

# Normalize categorical values
df = df.withColumn("gender", when(col("gender") == "Male", "M")
                                .when(col("gender") == "Female", "F")
                                .otherwise(col("gender")))

df = df.withColumn("attrition", when(col("attrition") == "Yes", 1)
                                   .otherwise(0))

df = df.withColumn("overtime", when(col("overtime") == "Yes", 1)
                                  .otherwise(0))

# Remove nulls
df = df.na.drop()

# Write to Silver
df.write.format("delta").mode("overwrite").save("Files/Silver/Silver_HR_Employee_Data")


StatementMeta(, 4f2557ca-358e-4ba7-add8-a3c815eadbda, 4, Finished, Available, Finished, False)